### TF-IDF Vectorization and Cosine Similarity for Search

This section demonstrates how to build a simple similarity search system using TF-IDF vectorization and cosine similarity. We will first create a sample dataset, vectorize the text data, and then define a function to find the most similar queries to a given input.

In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create a sample DataFrame with 'QueryText' and 'Category' columns
data = {
    'QueryText': [
        'How to make pasta?',
        'Best pasta recipes',
        'Italian pasta dishes',
        'Quick dinner ideas',
        'Healthy breakfast options',
        'What is a good breakfast?',
        'How to cook rice?',
        'Recipes for rice dishes',
        'How to bake a cake?',
        'Simple cake recipes'
    ],
    'Category': [
        'Cooking',
        'Cooking',
        'Cooking',
        'Food',
        'Food',
        'Food',
        'Cooking',
        'Cooking',
        'Baking',
        'Baking'
    ]
}
df = pd.DataFrame(data)

print("Sample DataFrame:")
display(df.head())


Sample DataFrame:


,QueryText,Category
0,How to make pasta?,Cooking
1,Best pasta recipes,Cooking
2,Italian pasta dishes,Cooking
3,Quick dinner ideas,Food
4,Healthy breakfast options,Food


#### TF-IDF Vectorization

We will now initialize the `TfidfVectorizer` and fit it to our `QueryText` data. This process converts text documents into a matrix of TF-IDF features.

In [2]:
# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the 'QueryText' column
tfidf_matrix = tfidf_vectorizer.fit_transform(df['QueryText'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Features (vocabulary):")
print(tfidf_vectorizer.get_feature_names_out()[:10]) # Display first 10 features


TF-IDF matrix shape: (10, 23)
Features (vocabulary):
['bake' 'best' 'breakfast' 'cake' 'cook' 'dinner' 'dishes' 'for' 'good'
 'healthy']


#### Similarity Search Function

Now, let's create a function that takes a new query, vectorizes it using the same TF-IDF model, and then calculates the cosine similarity with all existing queries in our dataset. It will return the top N most similar queries.

In [3]:
def find_similar_queries(query, top_n=3):
    # Transform the input query using the fitted TF-IDF vectorizer
    query_vec = tfidf_vectorizer.transform([query])

    # Calculate cosine similarity between the query and all documents
    cosine_similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Get the indices of the top_n most similar queries
    most_similar_indices = cosine_similarities.argsort()[-top_n:][::-1]

    # Get the similarity scores for the top_n queries
    top_n_scores = cosine_similarities[most_similar_indices]

    # Retrieve the actual queries and their scores
    similar_queries = df.loc[most_similar_indices, 'QueryText'].tolist()

    results = []
    for i in range(top_n):
        results.append({
            'query': similar_queries[i],
            'similarity_score': top_n_scores[i]
        })
    return results

# Demonstrate the similarity search
search_query = "I want to cook Italian food"
similar_results = find_similar_queries(search_query, top_n=2)

print(f"\nTop 2 similar queries for '{search_query}':")
for res in similar_results:
    print(f"- Query: '{res['query']}', Similarity: {res['similarity_score']:.4f}")

search_query_2 = "Breakfast recipe ideas"
similar_results_2 = find_similar_queries(search_query_2, top_n=3)

print(f"\nTop 3 similar queries for '{search_query_2}':")
for res in similar_results_2:
    print(f"- Query: '{res['query']}', Similarity: {res['similarity_score']:.4f}")



Top 2 similar queries for 'I want to cook Italian food':
- Query: 'How to cook rice?', Similarity: 0.5779
- Query: 'Italian pasta dishes', Similarity: 0.4149

Top 3 similar queries for 'Breakfast recipe ideas':
- Query: 'Quick dinner ideas', Similarity: 0.4399
- Query: 'Healthy breakfast options', Similarity: 0.3337
- Query: 'What is a good breakfast?', Similarity: 0.2854


#### Farmer Query Similarity Search Function

This function is designed to find the top N most similar past queries to a new farmer's query, including the category of each similar query.

In [5]:
def find_similar_farmer_queries(farmer_query, top_n=5):
    # Transform the input farmer query using the fitted TF-IDF vectorizer
    query_vec = tfidf_vectorizer.transform([farmer_query])

    # Calculate cosine similarity between the farmer query and all existing queries
    cosine_similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Get the indices of the top_n most similar queries
    # Use argsort() to get indices that would sort the array, then slice from end for top_n, and reverse to get highest first
    most_similar_indices = cosine_similarities.argsort()[-top_n:][::-1]

    # Retrieve the actual queries, their categories, and similarity scores
    results = []
    for i in most_similar_indices:
        results.append({
            'query': df.loc[i, 'QueryText'],
            'category': df.loc[i, 'Category'],
            'similarity_score': cosine_similarities[i]
        })
    return results

# Demonstrate the new farmer query similarity search
farmer_search_query = "What should I plant for breakfast?"
similar_farmer_results = find_similar_farmer_queries(farmer_search_query, top_n=5)

print(f"\nTop 5 similar queries for farmer's query: '{farmer_search_query}':")
for res in similar_farmer_results:
    print(f"- Query: '{res['query']}', Category: '{res['category']}', Similarity: {res['similarity_score']:.4f}")

farmer_search_query_2 = "How to make a delicious Italian dish?"
similar_farmer_results_2 = find_similar_farmer_queries(farmer_search_query_2, top_n=3)

print(f"\nTop 3 similar queries for farmer's query: '{farmer_search_query_2}':")
for res in similar_farmer_results_2:
    print(f"- Query: '{res['query']}', Category: '{res['category']}', Similarity: {res['similarity_score']:.4f}")



Top 5 similar queries for farmer's query: 'What should I plant for breakfast?':
- Query: 'What is a good breakfast?', Category: 'Food', Similarity: 0.5411
- Query: 'Recipes for rice dishes', Category: 'Cooking', Similarity: 0.3500
- Query: 'Healthy breakfast options', Category: 'Food', Similarity: 0.2654
- Query: 'Simple cake recipes', Category: 'Baking', Similarity: 0.0000
- Query: 'How to bake a cake?', Category: 'Baking', Similarity: 0.0000

Top 3 similar queries for farmer's query: 'How to make a delicious Italian dish?':
- Query: 'How to make pasta?', Category: 'Cooking', Similarity: 0.7328
- Query: 'Italian pasta dishes', Category: 'Cooking', Similarity: 0.3761
- Query: 'How to cook rice?', Category: 'Cooking', Similarity: 0.3732


#### Testing the Farmer Query Similarity Search with a New Query

Let's test the `find_similar_farmer_queries` function with the query: "How can I control pests in my tomato crop?"

In [6]:
new_farmer_query = "How can I control pests in my tomato crop?"
top_similar_pests_queries = find_similar_farmer_queries(new_farmer_query, top_n=5)

print(f"\nTop 5 similar queries for farmer's query: '{new_farmer_query}':")
for res in top_similar_pests_queries:
    print(f"- Query: '{res['query']}', Category: '{res['category']}', Similarity: {res['similarity_score']:.4f}")



Top 5 similar queries for farmer's query: 'How can I control pests in my tomato crop?':
- Query: 'How to make pasta?', Category: 'Cooking', Similarity: 0.4561
- Query: 'How to bake a cake?', Category: 'Baking', Similarity: 0.4422
- Query: 'How to cook rice?', Category: 'Cooking', Similarity: 0.4422
- Query: 'Simple cake recipes', Category: 'Baking', Similarity: 0.0000
- Query: 'What is a good breakfast?', Category: 'Food', Similarity: 0.0000
